# Week 8 — Files, Testing, and Documentation

**COMP 1110 — Introduction to Computer Programming**

**Tuesday, October 27 & Thursday, October 29, 2026**

Instructor: **Quan Nguyen** · Room **OM 1241** · 9:30 – 11:20 AM

*Tuesday: Files (Chapter 7) · Thursday: Testing & Documentation*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TRU-CS/COMP_1110/blob/main/COMP_1110/notebooks/week08_files_testing_docs.ipynb)

*Run this notebook yourself: press the power button at the top of the page to
execute cells right here, or click the badge to open an editable copy in Google
Colab (a Google account is required).*


## Where we are

- **Last week (7):** strings — indexing, slicing, methods, f-strings and output formatting.
- **This week (8):** *Tuesday* — reading and writing **files**.
  *Thursday* — **testing** your code and **documenting** it.
- **Next week (9):** lists — the natural way to hold the many lines you just read from a file.

## Today's outcomes

By the end of the week you should be able to:

1. Open a text file, read it line by line, and search it for the lines you care about. **[CLO 1]**
2. Handle a missing file with `try` / `except` instead of crashing. **[CLO 4]**
3. Write output to a new file. **[CLO 1]**
4. Write `assert` checks and a small `unittest.TestCase` for a function. **[CLO 4]**
5. Write a docstring that says what a function takes, returns, and does. **[CLO 2]**

## Reading

**Tuesday:** [Chapter 7 — Files](https://www.py4e.com/html3/07-files)

**Thursday** (not in the textbook — links are also on Moodle):

- [A Beginner's Guide to Unit Tests in Python](https://www.dataquest.io/blog/unit-tests-python/) (Dataquest)
- [Documenting Python Code: A Complete Guide](https://realpython.com/documenting-python-code/) (Real Python)

You are responsible for the `assert`, `unittest.TestCase`, and docstring sections only —
not `setUp`/`tearDown`, not Sphinx, not doctest.

## ⚠️ Yes, this week is on Midterm 2

**Midterm 2 — Thursday, November 19, 9:30–10:30, on Moodle, closed-book.**

Everything in Week 8 **is examinable**: files *and* testing and docstrings.
The syllabus lists Midterm 2 coverage as "Chapters 6–10 … **testing and docstrings**".

Expect to be asked to *read* file-handling code and state its output, and to say
what an `assert` or a docstring is for — with **no Python running**.

# Part 1 — Files

## Tuesday, October 27

Chapter 7: opening files, reading line by line, searching, missing files, writing files.

## Why files?

Everything we have written so far forgets everything the moment it stops.

- Variables live in memory; memory is wiped when the program ends.
- A **file** is data on disk that outlives the program.

Real data arrives as files: a class list, a temperature log, an export of your bank
transactions, a chat history. If your program cannot read a file, it can only work with
what someone types in by hand.

## First: we make our own data file

We are in a notebook, so there is no data file sitting next to us. Let's **write one**,
then spend the rest of the class **reading it back**.

`open(name, "w")` opens a file for **writing**. We will look at writing properly later —
for now just run this cell so the data exists.

In [1]:
grades_text = """Amina 88
Ben 72
Chi 95
Diego 64
Elena 79
"""

file_out = open("grades.txt", "w")
file_out.write(grades_text)
file_out.close()

print("grades.txt written")

grades.txt written


In [2]:
mail_text = """From: stephen@example.edu
Subject: Week 8 worksheet
From: marquard@example.edu
Subject: Files and testing
From: zqian@example.edu
Subject: Office hours
"""

file_out = open("mail_log.txt", "w")
file_out.write(mail_text)
file_out.close()

print("mail_log.txt written")

mail_log.txt written


## Opening a file

```python
file_handle = open("grades.txt")
```

- `open()` does **not** read the file. It hands you a **file handle**: a connection to the
  file, plus a bookmark for how far you have read.
- The name is a *relative* path — Python looks in the folder the program is running in.
- When you are done: `file_handle.close()`.

In [3]:
file_handle = open("grades.txt")
print(file_handle)
print(type(file_handle))
file_handle.close()

<_io.TextIOWrapper name='grades.txt' mode='r' encoding='UTF-8'>
<class '_io.TextIOWrapper'>


## A text file is a sequence of lines

On disk `grades.txt` is one long string. What makes it "five lines" is an invisible
character at the end of each one: the **newline**, written `\n`.

`"Amina 88\n" + "Ben 72\n" + ...`

`\n` is **one** character, not two — the backslash is just how we type it.

In [4]:
line = "Amina 88\n"
print(len(line))          # 8 visible characters + 1 newline
print(repr(line))         # repr() shows the escape characters
print(line[-1] == "\n")

9
'Amina 88\n'
True


## Reading a file line by line

The file handle is a **sequence of lines** — so a `for` loop walks it, exactly like a
`for` loop over a string walks characters.

```python
for line in file_handle:
    ...
```

This reads one line at a time. A 10 GB log file will not fill your memory.

In [5]:
file_handle = open("grades.txt")
count = 0
for line in file_handle:
    count = count + 1
file_handle.close()

print("Line count:", count)

Line count: 5


## Predict the output #1

```python
file_handle = open("grades.txt")
for line in file_handle:
    print(line)
file_handle.close()
```

The file has 5 lines. How many blank lines appear in the output, and why?

**Answer:** you get a blank line **after every one** of the 5 lines — the output is
double-spaced.

Each `line` still ends with its own `\n`, and `print()` adds a **second** newline.
Two newlines in a row = one blank line.

Fix: `print(line.rstrip())` or `print(line, end="")`.

## `.rstrip()` — the fix you will use constantly

`.rstrip()` returns a copy of the string with **whitespace on the right removed** —
including the trailing `\n`.

This matters for more than tidy printing: `"88\n"` is **not** equal to `"88"`, and
`int("88\n")` happens to work but `int(" 88 x")` does not. Strip first, compare second.

In [6]:
file_handle = open("grades.txt")
for line in file_handle:
    clean = line.rstrip()
    print(clean, "->", len(clean))
file_handle.close()

Amina 88 -> 8
Ben 72 -> 6
Chi 95 -> 6
Diego 64 -> 8
Elena 79 -> 8


## Reading the whole file at once

`file_handle.read()` returns the **entire** file as one big string.

Convenient for small files, dangerous for large ones — the whole file lands in memory.
Use it when you want to count characters or search the text as a whole.

In [7]:
file_handle = open("grades.txt")
whole = file_handle.read()
file_handle.close()

print(len(whole), "characters")
print(whole[:14])
print("Chi" in whole)

41 characters
Amina 88
Ben 7
True


## Searching through a file

The classic pattern: loop over every line, and use an `if` to keep only the lines you want.

```python
for line in file_handle:
    line = line.rstrip()
    if line.startswith("From:"):
        print(line)
```

`.startswith()` is a string method from last week. Files give you the lines; **strings**
give you the tools to inspect them.

In [8]:
file_handle = open("mail_log.txt")
for line in file_handle:
    line = line.rstrip()
    if line.startswith("From:"):
        print(line)
file_handle.close()

From: stephen@example.edu
From: marquard@example.edu
From: zqian@example.edu


### Skipping instead of nesting

Same result, "guard" style: if the line is *not* interesting, `continue` to the next one.
Keeps the interesting work un-indented.

In [9]:
file_handle = open("mail_log.txt")
found = 0
for line in file_handle:
    line = line.rstrip()
    if not line.startswith("From:"):
        continue
    found = found + 1
file_handle.close()

print(f"{found} From: lines")

3 From: lines


## Combining files with what we know

Read `grades.txt`, pull the name and the score out of each line, and average them.

Each line looks like `Amina 88`. There is a space in the middle, so:

- `line.find(" ")` gives the position of the space,
- slice before it for the name, slice after it for the score,
- `int()` turns the score text into a number.

In [10]:
file_handle = open("grades.txt")
total = 0
count = 0
for line in file_handle:
    line = line.rstrip()
    space = line.find(" ")
    name = line[:space]
    score = int(line[space + 1:])
    total = total + score
    count = count + 1
file_handle.close()

print(f"Average of {count} students: {total / count:.1f}")

Average of 5 students: 79.6


## Predict the output #2

```python
file_handle = open("grades.txt")
for line in file_handle:
    print(line.rstrip())

for line in file_handle:
    print("second loop:", line.rstrip())
file_handle.close()
```

What does the **second** loop print?

**Answer: nothing at all.**

The file handle keeps a bookmark. The first loop reads to the **end** of the file, and the
bookmark stays there — so the second loop has nothing left to read and its body never runs.

To read the file twice you must `open()` it again.

## Letting the user choose the file

```python
file_name = input("Enter a file name: ")
file_handle = open(file_name)
```

Now the program works on *any* file, not just the one you hard-coded.

(We cannot call `input()` in a notebook slideshow — it would block forever — so below we
assign the variable directly and pretend the user typed it.)

In [11]:
file_name = "mail_log.txt"    # imagine the user typed this at input()

file_handle = open(file_name)
count = 0
for line in file_handle:
    if line.startswith("Subject:"):
        count = count + 1
file_handle.close()

print(f"{file_name} has {count} Subject: lines")

mail_log.txt has 3 Subject: lines


## But users mistype file names

```python
file_name = "grads.txt"        # typo!
file_handle = open(file_name)
```

`open()` on a file that does not exist raises **`FileNotFoundError`** and your program
dies on the spot. Watch:

In [12]:
file_handle = open("grads.txt")

FileNotFoundError: [Errno 2] No such file or directory: 'grads.txt'

## `try` / `except` around `open()`

From Week 3: put the risky line in `try`, the recovery in `except`.

The rule: **keep the `try` block as small as possible** — just the `open()` call. If you
wrap the whole program, a bug anywhere silently becomes "file not found".

In [13]:
file_name = "grads.txt"       # imagine the user typed this

try:
    file_handle = open(file_name)
except FileNotFoundError:
    print(f"Sorry, I cannot open {file_name}")
    file_handle = None

if file_handle is not None:
    print(file_handle.read())
    file_handle.close()

Sorry, I cannot open grads.txt


### Ask again until it works

Loop-and-a-half from Week 5, with `try`/`except` inside. In a real program the commented
`input()` line is what you would use.

In [14]:
attempt = 0
while True:
    attempt = attempt + 1
    # file_name = input("Enter a file name: ")
    if attempt == 1:
        file_name = "grads.txt"       # first guess: a typo
    else:
        file_name = "grades.txt"      # second guess: correct
    try:
        file_handle = open(file_name)
        break
    except FileNotFoundError:
        print(f"No such file: {file_name} — try again")

print("Opened", file_name)
file_handle.close()

No such file: grads.txt — try again
Opened grades.txt


## Writing files

`open(name, "w")` opens for **writing**.

- The second argument is the **mode**: `"r"` read (the default), `"w"` write, `"a"` append.
- **`"w"` empties an existing file immediately.** Opening `grades.txt` with `"w"` by
  mistake destroys it. There is no undo.
- `.write()` writes exactly the string you give it — it does **not** add a newline. You must
  put `\n` in yourself.
- Always `.close()`; until you do, some of your text may still be sitting in a buffer.

In [15]:
file_handle = open("grades.txt")
report = open("honour_roll.txt", "w")
for line in file_handle:
    line = line.rstrip()
    score = int(line[line.find(" ") + 1:])
    if score >= 80:
        report.write(line + "\n")
file_handle.close()
report.close()

print(open("honour_roll.txt").read())

Amina 88
Chi 95



### `"a"` appends instead of erasing

Use `"a"` for logs — anything where you add to the end and keep what is already there.

In [16]:
log = open("honour_roll.txt", "a")
log.write("Ben 72 (added by appeal)\n")
log.close()

print(open("honour_roll.txt").read())

Amina 88
Chi 95
Ben 72 (added by appeal)



## Live coding — together

**Problem.** Read `grades.txt` and print the name of the student with the **highest**
score, and that score.

Reminders: open the file, loop, `rstrip()`, find the space, slice, `int()`. Keep a
"best so far" variable — the max pattern from Week 5.

In [17]:
# Find the top student in grades.txt
# Print e.g.  Top student: Chi with 95

In [18]:
# --- Solution ---
file_handle = open("grades.txt")
best_name = ""
best_score = -1
for line in file_handle:
    line = line.rstrip()
    space = line.find(" ")
    name = line[:space]
    score = int(line[space + 1:])
    if score > best_score:
        best_score = score
        best_name = name
file_handle.close()

print(f"Top student: {best_name} with {best_score}")

Top student: Chi with 95


## Common errors — files

| What you wrote | What Python says | What it means |
| :--- | :--- | :--- |
| `open("grads.txt")` | `FileNotFoundError: [Errno 2] No such file or directory: 'grads.txt'` | Wrong name, or the file is not in the folder you are running from. |
| `open("grades.txt").write("hi")` | `io.UnsupportedOperation: not writable` | You opened for reading. Writing needs mode `"w"` or `"a"`. |
| `for line in "grades.txt":` | *(no error — it loops over 10 characters!)* | You looped over the **file name**, not the file. You forgot `open()`. |
| `total = total + line` | `TypeError: unsupported operand type(s) for +: 'int' and 'str'` | A line is text. Convert with `int()` after `rstrip()`. |

## Part 1 summary

- `open(name)` returns a **file handle**, not the data. `close()` when done.
- A `for` loop over a handle gives you **one line at a time**, each ending in `\n`.
- `.rstrip()` removes that newline — do it before printing, comparing, or converting.
- Search a file with `for` + `if` + string methods like `.startswith()`.
- A handle can only be read **once**; re-`open()` to go again.
- `try` / `except FileNotFoundError` around `open()` keeps a typo from killing the program.
- `"w"` **erases**, `"a"` appends, and `.write()` does not add newlines for you.

# Part 2 — Testing & Documentation

## Thursday, October 29

Not in the textbook — see the Dataquest and Real Python readings.

Same question both halves: *how do I know this code is right, and how does the next
person understand it?*

## Why test at all?

"It ran and printed something" is not the same as "it is correct".

- You ran your function **once**, with one input, and eyeballed the answer.
- Then you changed a line to fix something else, and quietly broke the first thing.

A **test** is code that checks code. Written once, it re-checks every function every time
you run it — for free, forever. That is the whole idea.

> Your final project must ship *"a documented test plan with `assert`-based or `unittest`
> checks"*. This is that lecture.

## Start with a manual test plan

Before writing any test code, write down — in a table, in your README — what *should*
happen. For a function `letter_grade(score)`:

| Input | Expected | Why this case |
| :--- | :--- | :--- |
| `85` | `"A"` | typical |
| `80` | `"A"` | **boundary** — exactly on the line |
| `79` | `"B"` | boundary — just below |
| `0` | `"F"` | extreme but legal |
| `-5` | error / `"invalid"` | **invalid input** |

Most bugs live at the boundaries. `>` versus `>=` is the single most common off-by-one
mistake in this course.

## Here is the function we will test

Read it now and look for the boundary bug before we test for it.

In [19]:
def letter_grade(score):
    if score >= 80:
        return "A"
    elif score >= 70:
        return "B"
    elif score >= 60:
        return "C"
    return "F"


print(letter_grade(85), letter_grade(80), letter_grade(79), letter_grade(0))

A A B F


## `assert` — the one-line test

```python
assert condition, "message if it fails"
```

- Condition **True** → nothing happens at all. Silence means pass.
- Condition **False** → `AssertionError`, and your program stops right there.

`assert` is for checking things you believe are *always* true. It is not for validating
user input — for that you use `if` and `try`/`except`.

In [20]:
assert letter_grade(85) == "A", "85 should be an A"
assert letter_grade(80) == "A", "80 is the boundary of A"
assert letter_grade(79) == "B", "79 should be a B"
assert letter_grade(0) == "F", "0 should be an F"

print("All 4 assertions passed")

All 4 assertions passed


### What a failure looks like

Deliberately wrong expectation, so you recognise the message when it is *your* bug.

In [21]:
assert letter_grade(79) == "A", "79 should be a B, not an A"

AssertionError: 79 should be a B, not an A

## Predict the output #3

```python
def average_of_two(a, b):
    return a + b / 2

assert average_of_two(4, 4) == 4
print("passed")
```

Does this print `passed`?

**Answer: no.** `a + b / 2` is `4 + 2.0` = `6.0`, not `4`, so the assertion fails with
`AssertionError` and `print` never runs.

The missing parentheses — `(a + b) / 2` — is exactly the kind of bug an `assert` catches
and a single happy-looking test run does not. Precedence from Week 2, caught in Week 8.

## From `assert` to `unittest`

`assert` stops at the **first** failure, so you never see the other nine results.

`unittest` is in the standard library and fixes that:

- every check runs, and you get a **report**: how many ran, which failed, and why;
- tests are grouped into a class so they have names you can read;
- one command runs all of them.

## The shape of a `unittest` test

```python
import unittest

class TestLetterGrade(unittest.TestCase):
    def test_typical_score(self):
        self.assertEqual(letter_grade(85), "A")
```

Three rules you must not break:

1. The class **inherits from `unittest.TestCase`**.
2. Every test method's name **starts with `test`** — otherwise it is not collected.
3. Each method takes `self` and uses `self.assertEqual(...)`, not bare `assert`.

## Running it — inside a notebook

Normally you run `python -m unittest` in the terminal. In a notebook, use:

```python
unittest.main(argv=["ignored", "-v"], exit=False)
```

- `argv=[...]` hands `unittest` a fake command line; `-v` asks for one line per test.
- **`exit=False` matters** — without it `unittest` tries to exit Python and kills the
  notebook kernel.

(The square brackets are a *list*. That is next week — copy the line for now.)

In [22]:
import unittest


class TestLetterGrade(unittest.TestCase):

    def test_typical_score(self):
        self.assertEqual(letter_grade(85), "A")

    def test_boundary_80_is_a(self):
        self.assertEqual(letter_grade(80), "A")

    def test_boundary_79_is_b(self):
        self.assertEqual(letter_grade(79), "B")


unittest.main(argv=["ignored", "-v"], exit=False)

test_boundary_79_is_b (__main__.TestLetterGrade.test_boundary_79_is_b) ... 

ok


test_boundary_80_is_a (__main__.TestLetterGrade.test_boundary_80_is_a) ... 

ok


test_typical_score (__main__.TestLetterGrade.test_typical_score) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.001s

OK


## Reading the report

- `test_boundary_79_is_b (…) ... ok` — one line per test, in **alphabetical** order, not
  the order you wrote them. Tests must not depend on each other.
- `Ran 3 tests in 0.001s` then `OK` — everything passed.
- A failure prints `FAIL`, the method name, and `AssertionError: 'B' != 'A'` — **actual
  first, expected second**.

## `assertEqual` and `assertTrue`

| Method | Passes when |
| :--- | :--- |
| `self.assertEqual(a, b)` | `a == b` — your workhorse |
| `self.assertTrue(x)` | `x` is true |
| `self.assertFalse(x)` | `x` is false |

Prefer `assertEqual` where you can: when it fails it prints **both** values.
`assertTrue(letter_grade(85) == "A")` only tells you "False is not true".

In [23]:
class TestGradeChecks(unittest.TestCase):

    def test_zero_fails(self):
        self.assertEqual(letter_grade(0), "F")

    def test_returns_a_string(self):
        self.assertTrue(isinstance(letter_grade(72), str))

    def test_high_and_low_differ(self):
        self.assertFalse(letter_grade(95) == letter_grade(35))


unittest.main(argv=["ignored", "-v"], exit=False)

test_high_and_low_differ (__main__.TestGradeChecks.test_high_and_low_differ) ... 

ok


test_returns_a_string (__main__.TestGradeChecks.test_returns_a_string) ... 

ok


test_zero_fails (__main__.TestGradeChecks.test_zero_fails) ... 

...

test_boundary_80_is_a (__main__.TestLetterGrade.test_boundary_80_is_a) ... 

ok


test_typical_score (__main__.TestLetterGrade.test_typical_score) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK


## What makes a good test case?

For every function, ask for at least three kinds:

1. **Typical** — the ordinary case you had in mind. `letter_grade(85)`.
2. **Boundary** — exactly on the edge, and one step either side. `79`, `80`, `81`.
   Also: empty string, zero, one item, the last character.
3. **Invalid** — input the function should never get. `letter_grade(-5)`,
   `letter_grade("A")`. Decide what *should* happen, then test that it does.

Three focused tests with clear names beat twenty copies of the typical case.
A test named `test_1` tells you nothing when it fails at 2 a.m.

## Live coding — together

**Problem.** Write `is_passing(score)` returning `True` when `score >= 60`, then a
`unittest.TestCase` with three tests: typical pass, typical fail, and the **boundary** 60.

Then run it with the `argv` / `exit=False` line.

In [24]:
# 1. def is_passing(score): ...
# 2. class TestIsPassing(unittest.TestCase): ... three test methods
# 3. unittest.main(argv=["ignored", "-v"], exit=False)

In [25]:
# --- Solution ---
def is_passing(score):
    return score >= 60


class TestIsPassing(unittest.TestCase):

    def test_clear_pass(self):
        self.assertTrue(is_passing(85))

    def test_clear_fail(self):
        self.assertFalse(is_passing(41))

    def test_boundary_exactly_60(self):
        self.assertTrue(is_passing(60))


unittest.main(argv=["ignored", "-v"], exit=False)

test_high_and_low_differ (__main__.TestGradeChecks.test_high_and_low_differ) ... 

ok


test_returns_a_string (__main__.TestGradeChecks.test_returns_a_string) ... 

ok


test_zero_fails (__main__.TestGradeChecks.test_zero_fails) ... 

...

test_boundary_80_is_a (__main__.TestLetterGrade.test_boundary_80_is_a) ... 

ok


test_typical_score (__main__.TestLetterGrade.test_typical_score) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.002s

OK


# Documentation

## Tests say the code works. Documentation says what it is *for*.

Both are graded on your final project.

## Comments vs docstrings

**A comment** (`#`) explains a *line* to whoever is editing the code. It is invisible at
runtime.

**A docstring** is a string on the first line of a function. It explains the *function* to
whoever is calling it — and it is a real object Python keeps, so `help()` can print it.

Rule of thumb: a comment says **why**; a docstring says **what**.
A comment that repeats the code (`x = x + 1  # add 1 to x`) is worse than no comment.

## Writing a function docstring

Triple quotes, immediately under the `def`:

1. **One-line summary**, imperative mood, ending in a period. `"Return the letter grade."`
2. Blank line.
3. **Args:** each parameter, its type, its meaning.
4. **Returns:** the type and what it means.

Keep it short. If the docstring needs three paragraphs, the function is doing too much.

In [26]:
def letter_grade(score):
    """Return the letter grade for a numeric score.

    Args:
        score (int): a percentage from 0 to 100.

    Returns:
        str: one of "A", "B", "C" or "F".
    """
    if score >= 80:
        return "A"
    elif score >= 70:
        return "B"
    elif score >= 60:
        return "C"
    return "F"

## `help()` and `__doc__`

The docstring is stored on the function as `__doc__`. That is why `help(letter_grade)`
works on *your* function the same way `help(len)` works on a built-in.

Every library you have ever used documents itself this way.

In [27]:
help(letter_grade)

Help on function letter_grade in module __main__:

letter_grade(score)
    Return the letter grade for a numeric score.

    Args:
        score (int): a percentage from 0 to 100.

    Returns:
        str: one of "A", "B", "C" or "F".



In [28]:
print(letter_grade.__doc__[:44])
print(len("---"), "| built-ins have docstrings too:")
print(str.rstrip.__doc__[:60])

Return the letter grade for a numeric score.
3 | built-ins have docstrings too:
Return a copy of the string with trailing whitespace removed


## What belongs in a project README

Your final project README is marked. It should contain:

1. **What the program does** — two or three sentences, no jargon.
2. **How to run it** — the exact command, and what file(s) it expects.
3. **Inputs and outputs** — a sample run, pasted.
4. **How it is organised** — the main functions and what each is responsible for.
5. **Test plan** — what you tested and how to run the tests.
6. **Known limitations** — what it does *not* handle. This earns marks; hiding it does not.
7. **Individual contribution statement** — who wrote which functions. *(Required.)*
8. **Citations** — any code adapted from a tutorial or Stack Overflow.

## Common errors — testing & documentation

| What you wrote | What happens | What it means |
| :--- | :--- | :--- |
| `unittest.main()` in a notebook | `SystemExit: True` / dead kernel | Use `unittest.main(argv=["ignored", "-v"], exit=False)`. |
| `def check_boundary(self):` | `Ran 0 tests in 0.000s` — **OK** | The name must start with `test`. A green run of zero tests is a trap. |
| `def test_x():` (no `self`) | `TypeError: test_x() takes 0 positional arguments but 1 was given` | Test methods live in a class; they take `self`. |
| `assertEquals` | `AttributeError` / deprecation | The method is `assertEqual` — no "s". |
| `'''docs'''` placed *above* the `def` | `help()` shows nothing | The docstring goes **inside** the function, as its first line. |

## Part 2 summary

- Tests are code that checks code — write them once, run them forever.
- Plan tests first: **typical**, **boundary**, **invalid**. Boundaries hide the bugs.
- `assert condition, "message"` is the one-line test; it stops at the first failure.
- `unittest.TestCase` runs them all and reports; methods must start with `test`.
- In a notebook: `unittest.main(argv=["ignored", "-v"], exit=False)`.
- Comments explain *why* a line exists; docstrings explain *what* a function does.
- `help()` reads your docstring straight out of `__doc__`. Every project function gets one.

## This week's worksheet — W7

**Released:** Tuesday Oct 27 at 9:00 AM · **Due:** Monday Nov 2, 11:59 PM, on the
**Practice Platform**.

Covers both halves:

- Read a file the worksheet tells you to create, count matching lines, and report an average.
- Wrap the `open()` in `try`/`except` so a bad name prints a message instead of crashing.
- Write one small function with a docstring, plus a `unittest.TestCase` with a typical, a
  boundary, and an invalid-input test.
- One trace-and-explain question on a file loop — no running it.

Marked for correctness **and** readability. Remember your lowest worksheet is dropped.

## Next class

**Tuesday, November 3 — Chapter 8: Lists.**

Files hand you lines one at a time and then forget them. Next week you get a container to
**keep** them in — and `line.split()`, which turns `"Amina 88"` into its pieces without
any `find()` and slicing.

Also Thursday Nov 5: **team project proposals due.**

Read [Chapter 8 — Lists](https://www.py4e.com/html3/08-lists) before Tuesday.